In [1]:
import pandas as pd
import os
import glob

# Month Wise PO's Read

In [2]:
folder_path = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\Quick_Commerece_Input_ALL\Zepto\Month_Wise_PO's"

dfs = []

for file in os.listdir(folder_path):
    if file.lower().endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        print(f"📥 Reading: {file}")
        temp_df = pd.read_csv(file_path)
        dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print(f"✅ Combined DataFrame shape: {df.shape}")

📥 Reading: 1jan_to_31jan.csv
📥 Reading: april.csv
📥 Reading: DEC_PO.csv
📥 Reading: feb2026.csv
📥 Reading: JUNE.csv
📥 Reading: MARCH_PO.csv
📥 Reading: MAY.csv
✅ Combined DataFrame shape: (4275, 27)


In [3]:
is_unique = not df.duplicated(subset=['PO No.', 'SKU']).any()

print("Is PO No + SKU combination unique?", is_unique)

Is PO No + SKU combination unique? True


In [4]:
df

,PO No.,PO Date,Status,Vendor Code,Vendor Name,PO Amount,Del Location,Line No,SKU,SKU Code,...,CESS %,MRP/RSP,Qty,Unit Base Cost,Landing Cost,Total Amount,Created By,ASN Quantity,GRN Quantity,PO Expiry Date
0,P3138482,02 Jan 2026 06:31 pm,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,39262.744229,MUM-SS-MH-SHAKTI,NaN,A518FA1D-8626-4CDF-AE76-F50B472CD99B,NaN,...,0,149,300,54.53,64.35,19305.00,naseem.khan@zeptonow.com,300,300,06 Feb 2026 11:59 pm
1,P3138482,02 Jan 2026 06:31 pm,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,39262.744229,MUM-SS-MH-SHAKTI,NaN,03937A89-E82F-4482-BC4A-EC53499C8168,NaN,...,0,159,200,63.27,66.43,13286.00,naseem.khan@zeptonow.com,200,200,06 Feb 2026 11:59 pm
2,P3138482,02 Jan 2026 06:31 pm,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,39262.744229,MUM-SS-MH-SHAKTI,NaN,DFAF5B5A-75F3-4A07-AD8F-63CC001BD2AA,NaN,...,0,229,100,86.46,90.78,9078.00,naseem.khan@zeptonow.com,100,100,06 Feb 2026 11:59 pm
3,P3138482,02 Jan 2026 06:31 pm,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,39262.744229,MUM-SS-MH-SHAKTI,NaN,A9AAC621-BA27-434C-9B73-04B6E2A3CF8E,NaN,...,0,129,15,38.01,44.85,672.75,naseem.khan@zeptonow.com,15,0,06 Feb 2026 11:59 pm
4,P3138482,02 Jan 2026 06:31 pm,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,39262.744229,MUM-SS-MH-SHAKTI,NaN,67795542-BDEB-409C-BFD7-7F13C5A30A4D,NaN,...,0,199,10,103.34,103.34,1033.40,naseem.khan@zeptonow.com,10,10,06 Feb 2026 11:59 pm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4270,P4503041,30 May 2026 07:38 am,PENDING_ACKNOWLEDGEMENT,KK-33068,ECOSOUL HOME PRIVATE LIMITED,46101.400000,MUM-DRY-MH-SHAKTI,NaN,0B2680CF-C341-4D7F-84F3-D6B78C86F7A5,NaN,...,0,149,80,60.04,70.85,5667.78,SYSTEM,0,0,06 Jul 2026 11:59 pm
4271,P4503041,30 May 2026 07:38 am,PENDING_ACKNOWLEDGEMENT,KK-33068,ECOSOUL HOME PRIVATE LIMITED,46101.400000,MUM-DRY-MH-SHAKTI,NaN,800E08DF-4BB0-489E-9D3A-6E5F817CB919,NaN,...,0,119,40,54.53,64.35,2573.82,SYSTEM,0,0,06 Jul 2026 11:59 pm
4272,P4503041,30 May 2026 07:38 am,PENDING_ACKNOWLEDGEMENT,KK-33068,ECOSOUL HOME PRIVATE LIMITED,46101.400000,MUM-DRY-MH-SHAKTI,NaN,94B0AE0E-75E3-4853-AB5B-B01FFBE1E2DF,NaN,...,0,169,150,69.11,72.57,10884.83,SYSTEM,0,0,06 Jul 2026 11:59 pm
4273,P4503041,30 May 2026 07:38 am,PENDING_ACKNOWLEDGEMENT,KK-33068,ECOSOUL HOME PRIVATE LIMITED,46101.400000,MUM-DRY-MH-SHAKTI,NaN,C5101EAA-ACF1-4AE1-BF95-0B8B2FA9EF84,NaN,...,0,89,200,49.02,57.84,11568.72,SYSTEM,0,0,06 Jul 2026 11:59 pm


# Remove Duplication by PO + SKU

In [5]:
import pandas as pd

# ===============================
# STEP 1: Parse PO Expiry Date with explicit format
# ===============================
df["PO Expiry Date"] = pd.to_datetime(
    df["PO Expiry Date"],
    format="%d %b %Y %I:%M %p",
    errors="coerce"
)

# ===============================
# STEP 2: Sort so latest comes first
# ===============================
df = df.sort_values(
    by=["PO No.", "SKU", "PO Expiry Date"],
    ascending=[True, True, False]
)

# ===============================
# STEP 3: Drop duplicates (keep latest expiry)
# ===============================
df = df.drop_duplicates(
    subset=["PO No.", "SKU"],
    keep="first"
).reset_index(drop=True)

print(f"✅ Final rows after deduplication: {df.shape[0]}")


✅ Final rows after deduplication: 4275


In [6]:
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta

# ===============================
# STEP 1: Parse PO Date (explicit format)
# ===============================
df["PO Date"] = pd.to_datetime(
    df["PO Date"],
    format="%d %b %Y %I:%M %p",
    errors="coerce"
)


# ===============================
# STEP 2: Calculate cutoff date (current month - 2)
# ===============================
today = date.today()

cutoff_date = (
    today.replace(day=1)          # first day of current month
    - relativedelta(months=2)     # go back 2 months
)

print(f"📅 Keeping data from: {cutoff_date}")

# ===============================
# STEP 3: Filter dataframe
# ===============================
df = df[df["PO Date"] >= pd.Timestamp(cutoff_date)].reset_index(drop=True)

print(f"✅ Rows after filtering: {df.shape[0]}")


📅 Keeping data from: 2026-04-01
✅ Rows after filtering: 1644


In [7]:
df

,PO No.,PO Date,Status,Vendor Code,Vendor Name,PO Amount,Del Location,Line No,SKU,SKU Code,...,CESS %,MRP/RSP,Qty,Unit Base Cost,Landing Cost,Total Amount,Created By,ASN Quantity,GRN Quantity,PO Expiry Date
0,P3979255,2026-04-01 08:47:00,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,28477.90,CHN-DRY-MH2-THIRUVALLUR,NaN,0B2680CF-C341-4D7F-84F3-D6B78C86F7A5,NaN,...,0,149,40,60.04,70.85,2833.89,SYSTEM,40,40,2026-04-29 23:59:00
1,P3979255,2026-04-01 08:47:00,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,28477.90,CHN-DRY-MH2-THIRUVALLUR,NaN,3FF23046-B471-475D-8927-8E30F726979A,NaN,...,0,169,80,83.85,83.85,6708.00,SYSTEM,80,80,2026-04-29 23:59:00
2,P3979255,2026-04-01 08:47:00,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,28477.90,CHN-DRY-MH2-THIRUVALLUR,NaN,94B0AE0E-75E3-4853-AB5B-B01FFBE1E2DF,NaN,...,0,169,50,69.11,72.57,3628.28,SYSTEM,50,50,2026-04-29 23:59:00
3,P3979255,2026-04-01 08:47:00,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,28477.90,CHN-DRY-MH2-THIRUVALLUR,NaN,958BA1C5-D0A9-465A-9E1B-925D3A9FEEB3,NaN,...,0,229,80,98.12,103.03,8242.08,SYSTEM,80,80,2026-04-29 23:59:00
4,P3979255,2026-04-01 08:47:00,COMPLETED,KK-33068,ECOSOUL HOME PRIVATE LIMITED,28477.90,CHN-DRY-MH2-THIRUVALLUR,NaN,C5101EAA-ACF1-4AE1-BF95-0B8B2FA9EF84,NaN,...,0,89,120,49.02,57.84,6941.23,SYSTEM,120,120,2026-04-29 23:59:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1639,P4614376,2026-06-11 06:23:00,PENDING_ACKNOWLEDGEMENT,KK-33068,ECOSOUL HOME PRIVATE LIMITED,49094.06,MUM-DRY-MH3,NaN,800E08DF-4BB0-489E-9D3A-6E5F817CB919,117043.0,...,0,119,40,54.53,64.35,2573.82,SYSTEM,0,0,2026-07-18 23:59:00
1640,P4614376,2026-06-11 06:23:00,PENDING_ACKNOWLEDGEMENT,KK-33068,ECOSOUL HOME PRIVATE LIMITED,49094.06,MUM-DRY-MH3,NaN,94B0AE0E-75E3-4853-AB5B-B01FFBE1E2DF,117501.0,...,0,169,150,69.11,72.57,10884.83,SYSTEM,0,0,2026-07-18 23:59:00
1641,P4614376,2026-06-11 06:23:00,PENDING_ACKNOWLEDGEMENT,KK-33068,ECOSOUL HOME PRIVATE LIMITED,49094.06,MUM-DRY-MH3,NaN,C5101EAA-ACF1-4AE1-BF95-0B8B2FA9EF84,117425.0,...,0,89,40,49.02,57.84,2313.74,SYSTEM,0,0,2026-07-18 23:59:00
1642,P4614376,2026-06-11 06:23:00,PENDING_ACKNOWLEDGEMENT,KK-33068,ECOSOUL HOME PRIVATE LIMITED,49094.06,MUM-DRY-MH3,NaN,DF871848-E955-4F51-8703-BB19EF5A5FE9,117286.0,...,0,149,100,57.45,60.32,6032.25,SYSTEM,0,0,2026-07-18 23:59:00


## Rename and take only required columns in PO


In [8]:
import pandas as pd
import numpy as np
# Step 1: Rename columns
df = df.rename(columns={
    'MRP/RSP':'PO MRP',
    'PO No.': 'PO Number',
    'PO Date': 'PO Creation Date',
    'Vendor Name': 'PO Delivered By Address',
    'Del Location': 'PO Delivered To',
    'SKU': 'PO Item Code',
    'HSN': 'HSN Code',
    'EAN': 'PO Product UPC',
    'SKU Desc': 'PO Product Description',
    'Unit Base Cost': 'PO Basic Cost Price',
    'Landing Cost': 'PO Landing Rate',
    'Qty': 'PO_quantity',
    'Total Amount': 'PO_Amount'
})
df['PO Tax Amt'] = np.where(
    df['IGST %'].notna(),
    df['PO_Amount'] * df['IGST %'] / 100,
    df['PO_Amount'] * (df['SGST %'] + df['CGST %']) / 100
)

# Step 2: Select only required columns
required_columns = [
    'PO Number',
    'PO Creation Date',
    'PO Expiry Date',
    'PO Delivered By Address',
    'PO Delivered To',
    'PO Item Code',
    'HSN Code',
    'PO Product UPC',
    'PO Product Description',
    'PO Basic Cost Price',
    'PO Tax Amt',
    'PO Landing Rate',
    'PO_quantity',
    'PO_Amount',
    'PO MRP'
]

df = df[required_columns]


### Change the format of PO Creation Date and PO Amount

In [9]:
import pandas as pd

# Convert PO Creation Date to datetime and keep only date
df['PO Creation Date'] = pd.to_datetime(
    df['PO Creation Date'],
    format='%d %b %Y %I:%M %p',
    errors='coerce'
).dt.date

# Convert PO_Amount to numeric and round to 2 decimals
df['PO_Amount'] = pd.to_numeric(df['PO_Amount'], errors='coerce').round(2)


# Add SKU details

In [10]:
# Remove spaces inside Item Code values
df["PO Item Code"] = (
    df["PO Item Code"]
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str.strip().str.lower()
)
import pandas as pd

helper_file = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\helper_file.xlsx"

helper_df = pd.read_excel(helper_file, dtype=str)
helper_df.columns = helper_df.columns.str.strip()
helper_df["SKU Code"] = helper_df["SKU Code"].astype(str).str.strip()
helper_df["SKU"] = helper_df["SKU"].astype(str).str.strip()
# Create empty SKU column first
df["SKU"] = ""
itemcode_to_sku = helper_df.set_index("SKU Code")["SKU"].to_dict()

df["SKU"] = df["PO Item Code"].map(itemcode_to_sku).fillna("")
missing_item_codes = (
    df.loc[df["SKU"] == "", "PO Item Code"]
    .unique()
    .tolist()
)
sku_to_category = helper_df.set_index("SKU")["Category"].to_dict()
sku_to_box_case = helper_df.set_index("SKU")["Box/Case"].to_dict()
sku_to_material = helper_df.set_index("SKU")["Material"].to_dict()
sku_to_subcategory = helper_df.set_index("SKU")["Sub Category"].to_dict()
df["Category"] = df["SKU"].map(sku_to_category).fillna("")
df["Material"] = df["SKU"].map(sku_to_material).fillna("")
df["Sub Category"] = df["SKU"].map(sku_to_subcategory).fillna("")
df["Box/Case"] = df["SKU"].map(sku_to_box_case).fillna("")
missing_item_codes

[]

In [11]:
df_po_month=df.copy()

In [12]:
df_po_month

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO Landing Rate,PO_quantity,PO_Amount,PO MRP,SKU,Category,Material,Sub Category,Box/Case
0,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,0b2680cf-c341-4d7f-84f3-d6b78c86f7a5,48236900,810103154292,ECO SOUL | Disposable Paper Cups/Glasses | 180...,60.04,510.1002,70.85,40,2833.89,149,CRC6OZ25NL,Home Essentials,Paper,Kitchen Essentials,40
1,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,3ff23046-b471-475d-8927-8e30f726979a,46021919,810103152274,ECO SOUL | Palm Leaf Disposable Plates | 8 Inc...,83.85,0.0000,83.85,80,6708.00,169,PLP8RO10,Home Essentials,Palm Leaf,Tableware,20
2,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,181.4140,72.57,50,3628.28,169,CNB3CMT10UB,Home Essentials,Bagasse,Tableware,50
3,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,958ba1c5-d0a9-465a-9e1b-925d3a9feeb3,48237010,810103158153,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,98.12,412.1040,103.03,80,8242.08,229,CNB5CMT10UB,Home Essentials,Bagasse,Tableware,20
4,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,1249.4214,57.84,120,6941.23,89,PCL4OZ50NL,Home Essentials,PLA,Kitchen Essentials,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1639,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,800e08df-4bb0-489e-9d3a-6e5f817cb919,48236900,810103156807,ECO SOUL | Disposable Paper Cups/Glasses | 180...,54.53,463.2876,64.35,40,2573.82,119,PCL6OZ25NL,Home Essentials,Paper,Tableware,40
1640,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,544.2415,72.57,150,10884.83,169,CNB3CMT10UB,Home Essentials,Bagasse,Tableware,50
1641,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,416.4732,57.84,40,2313.74,89,PCL4OZ50NL,Home Essentials,PLA,Kitchen Essentials,40
1642,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,df871848-e955-4f51-8703-bb19ef5a5fe9,44219990,810103151604,Eco Soul 160 mm | 50 Count | Disposable Spoon ...,57.45,301.6125,60.32,100,6032.25,149,BS50,Home Essentials,Birchwood,Tableware,100


# Month Wise Invoice Read

In [13]:
import os
import pandas as pd

folder_path = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\Quick_Commerece_Input_ALL\All_invoices"

dfs = []

for file in os.listdir(folder_path):
    if file.lower().endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        print(f"📥 Reading: {file}")
        temp_df = pd.read_csv(file_path)
        dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print(f"✅ Combined DataFrame shape: {df.shape}")


📥 Reading: 1DEC_TO_FEB.csv


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_21560\1186993499.py:12: DtypeWarning: Columns (74,87,89,98,101,126,144,145,170,171,172,173,180) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


📥 Reading: 1feb_to_april.csv


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_21560\1186993499.py:12: DtypeWarning: Columns (44,87,89,95,98,144,145,170,171,172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


📥 Reading: 1JAN_TO_MARCH.csv


...

📥 Reading: May25-july25.csv


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_21560\1186993499.py:12: DtypeWarning: Columns (26,44,55,74,82,84,88,90,96,115,116,117,118,121,123,124,126,130,132,134,135,136,138,139,140,141,153,155,158,172,173,174,176,177,179,182,183) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


📥 Reading: Nov24-jan25.csv
📥 Reading: Nov25-jan26.csv


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_21560\1186993499.py:12: DtypeWarning: Columns (26,44,55,74,82,84,88,90,93,96,115,116,117,118,121,123,124,126,132,134,135,136,138,139,140,141,158,167,172,173,174,175,182) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


✅ Combined DataFrame shape: (166083, 192)


In [14]:
df

,Invoice Date,Invoice ID,Invoice Number,Invoice Status,Invoice Type,Customer Name,Customer ID,Customer Number,Location ID,Location Name,...,CF.Dispatch From,CF.Shipping Charges,CF.Bookmark this invoice,LINEITEM.TAG.Branch,LINEITEM.TAG.Branch-Maharashtra,LINEITEM.TAG.Customer,LINEITEM.TAG.Geography,LINEITEM.TAG.Location,TAG.MONTHLY EXP,TAG.TAG
0,2026-01-01,1.051890e+18,FK/2025-26/094,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-01,1.051890e+18,FK/2025-26/094,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-01-01,1.051890e+18,FK/2025-26/094,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-01-01,1.051890e+18,FK/2025-26/095,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-01-01,1.051890e+18,FK/2025-26/096,Closed,Invoice,FLIPKART INDIA PVT LTD,1.051890e+18,CUS-00118,1.051890e+18,HO Gautam Buddha Nagar,...,"EcoSoul Home Pvt Ltd Plot 305, Udyog Kendra Ex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166078,2026-01-31,1.051890e+18,Amazon/245,Overdue,Invoice,Amazon Seller Services Pvt Ltd,1.051890e+18,CUS-00003,1.051890e+18,Branch-Telangana,...,Amazon Inventory Warehouse Telangana,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166079,2026-01-31,1.051890e+18,Amazon/245,Overdue,Invoice,Amazon Seller Services Pvt Ltd,1.051890e+18,CUS-00003,1.051890e+18,Branch-Telangana,...,Amazon Inventory Warehouse Telangana,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166080,2026-01-31,1.051890e+18,Amazon/245,Overdue,Invoice,Amazon Seller Services Pvt Ltd,1.051890e+18,CUS-00003,1.051890e+18,Branch-Telangana,...,Amazon Inventory Warehouse Telangana,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166081,2026-01-31,1.051890e+18,Amazon/245,Overdue,Invoice,Amazon Seller Services Pvt Ltd,1.051890e+18,CUS-00003,1.051890e+18,Branch-Telangana,...,Amazon Inventory Warehouse Telangana,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Select only zepto invoice

In [15]:
df = df[
    df['Customer Name']
    .astype(str)
    .str.strip()
    .str.upper()
    .eq('ZEPTO LIMITED')
]


# Select required columns and rename

In [16]:
import pandas as pd

# 1️⃣ Rename columns
df = df.rename(columns={
    'PurchaseOrder': 'PO Number',
    'Billing Address': 'invoice_bill_to',
    'Shipping Address': 'invoice_ship_to',
    'CF.Dispatch From': 'invoice_dispatch_from',
    'Item Name': 'Invoice Item & Description',
    'Quantity': 'Invoice_Quantity',
    'Item Total': 'Invoice_Amount',
    'Item Price': 'Invoice Rate'
})

# 2️⃣ Split Item Tax % into SGST % and CGST %
df['Item Tax %'] = pd.to_numeric(df['Item Tax %'], errors='coerce')
df['invoice_amt_gst'] = df['Invoice_Amount'] + df['Item Tax Amount']
df['SGST %'] = df['Item Tax %'] / 2
df['CGST %'] = df['Item Tax %'] / 2

# 3️⃣ Split Item Tax Amount into SGST Amt and CGST Amt
df['Item Tax Amount'] = pd.to_numeric(df['Item Tax Amount'], errors='coerce')

df['SGST Amt'] = df['Item Tax Amount'] / 2
df['CGST Amt'] = df['Item Tax Amount'] / 2

# 4️⃣ Create blank IGST columns
df['IGST %'] = None
df['IGST Amt'] = None

# 5️⃣ Drop original tax columns
df = df.drop(columns=['Item Tax %', 'Item Tax Amount'])

# 6️⃣ Keep only required columns
required_columns = [
    'Invoice Number',
    'PO Number',
    'Invoice Date',
    'SKU',
    'invoice_bill_to',
    'invoice_ship_to',
    'invoice_dispatch_from',
    'Invoice Item & Description',
    'Invoice_Quantity',
    'Invoice Rate',
    'Invoice_Amount',
    'invoice_amt_gst',
    'SGST %',
    'CGST %',
    'IGST %',
    'SGST Amt',
    'CGST Amt',
    'IGST Amt'
]

df = df[required_columns]


In [17]:
import regex as re
def clean_sku(s):
    """
    Clean SKU number by removing prefixes, spaces, and special characters.
    """
    if not s:
        return ""

    s = str(s).upper()

    # Remove leading prefixes (ZB, SP etc.)
    s = re.sub(r"^(ZB|SP)[\s\-\.\_]+", "", s)

    # Remove spaces/hyphens/dots
    s = re.sub(r"[\s\-\.\_]", "", s)

    # Allow only A–Z and 0–9
    s = re.sub(r"[^A-Z0-9]", "", s)

    return s.strip()


# Apply cleaning to SKU column
if 'SKU' in df.columns:
    print(f"Cleaning SKU column...")
    print(f"Before: Sample SKUs = {df['SKU'].head(10).tolist()}")

    # Create a new column with cleaned SKU or keep original if cleaning results in empty
    df['SKU Cleaned'] = df['SKU'].apply(clean_sku)

    # Replace original SKU with cleaned version (or keep original if cleaned is empty)
    df['SKU'] = df.apply(
        lambda row: row['SKU Cleaned'] if row['SKU Cleaned'] else row['SKU'],
        axis=1
    )

    # Drop the temporary cleaned column
    df = df.drop(columns=['SKU Cleaned'])

    print(f"After: Sample SKUs = {df['SKU'].head(10).tolist()}")
    print(f"✅ SKU column cleaned successfully!")
else:
    print("⚠️ 'SKU' column not found in dataframe")

df


Cleaning SKU column...
Before: Sample SKUs = ['ZB-CBKPT2P60P02', 'ZB-CBP10RO3C10BL', 'SP-PLP12RO10', 'ZB-CBTP3P160P02', 'ZB-CTBL15', 'ZB-CBDN1P100P01', 'ZB-BSP50', 'ZB-CBTP3P160P06', 'ZB-CBDN1P100P01', 'ZB-CBP10RO3C10BL']
After: Sample SKUs = ['CBKPT2P60P02', 'CBP10RO3C10BL', 'PLP12RO10', 'CBTP3P160P02', 'CTBL15', 'CBDN1P100P01', 'BSP50', 'CBTP3P160P06', 'CBDN1P100P01', 'CBP10RO3C10BL']
✅ SKU column cleaned successfully!


,Invoice Number,PO Number,Invoice Date,SKU,invoice_bill_to,invoice_ship_to,invoice_dispatch_from,Invoice Item & Description,Invoice_Quantity,Invoice Rate,Invoice_Amount,invoice_amt_gst,SGST %,CGST %,IGST %,SGST Amt,CGST Amt,IGST Amt
59,ZO/2025-26/047,P2666359,2025-12-01,CBKPT2P60P02,"BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Bagasse Kitchen Paper 2 Ply - 60 P...,108.0,82.07,8863.56,10459.00,9.0,9.0,None,797.72,797.72,None
60,ZO/2025-26/047,P2666359,2025-12-01,CBP10RO3C10BL,"BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...","Compostable Bagasse Plates, 10 inch-3 Compartm...",100.0,63.27,6327.00,6643.36,2.5,2.5,None,158.18,158.18,None
61,ZO/2025-26/047,P2666359,2025-12-01,PLP12RO10,"BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...","Palm Leaf Plates - Round, 12 inch, 10 count",100.0,109.85,10985.00,10985.00,0.0,0.0,None,0.00,0.00,None
62,ZO/2025-26/047,P2666359,2025-12-01,CBTP3P160P02,"BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Compostable Bagasse Tissue Paper 3 Ply - 160 P...,100.0,54.54,5454.00,6435.72,9.0,9.0,None,490.86,490.86,None
63,ZO/2025-26/047,P2666359,2025-12-01,CTBL15,"BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","BLR-SS-MH-SUMADHURA (BLR135M) \nBlock -D, Suma...","Ecosoul Home Pvt Ltd 1st Phase, Plot No. 162-A...",Large PLA Trash Bag - - Count of 15 - Green,30.0,114.33,3429.90,4047.28,9.0,9.0,None,308.69,308.69,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151133,ZO/2025-26/292,P3186238,2026-01-30,CBTP3P160P02,MUM-SS-MH-SHAKTI (MUM175M),MUM-SS-MH-SHAKTI (MUM175M),"Ecosoul Private Limited Waredepot, C/O Top Ind...",Compostable Bagasse Tissue Paper 3 Ply - 160 P...,300.0,54.52,16356.00,19300.08,9.0,9.0,None,1472.04,1472.04,None
151134,ZO/2025-26/292,P3186238,2026-01-30,CPS6MM50NB,MUM-SS-MH-SHAKTI (MUM175M),MUM-SS-MH-SHAKTI (MUM175M),"Ecosoul Private Limited Waredepot, C/O Top Ind...",6 mm Paper Straw - - Count of 50 - White,200.0,26.99,5398.00,6369.64,9.0,9.0,None,485.82,485.82,None
151135,ZO/2025-26/292,P3186238,2026-01-30,CPHC5OZ50NL,MUM-SS-MH-SHAKTI (MUM175M),MUM-SS-MH-SHAKTI (MUM175M),"Ecosoul Private Limited Waredepot, C/O Top Ind...",5 Oz/ 150 ml Paper Cup - Count of 50- Multicolor,120.0,54.57,6548.40,7727.12,9.0,9.0,None,589.36,589.36,None
151136,ZO/2025-26/292,P3186238,2026-01-30,PLP10RO10,MUM-SS-MH-SHAKTI (MUM175M),MUM-SS-MH-SHAKTI (MUM175M),"Ecosoul Private Limited Waredepot, C/O Top Ind...","Palm Leaf Plates10"" RD- Shrink Pack of 10",90.0,103.34,9300.60,9300.60,0.0,0.0,None,0.00,0.00,None


In [18]:
invoice_df_month= df.copy()

In [19]:
duplicate_count = invoice_df_month.duplicated(subset=['PO Number', 'SKU']).sum()
print("Number of duplicate rows:", duplicate_count)


Number of duplicate rows: 8314


In [20]:
invoice_df_month.drop_duplicates(subset=['SKU', 'PO Number'], inplace=True)


# PO Invoice merge

In [21]:
merged_df = df_po_month.merge(
    invoice_df_month,
    on=['PO Number', 'SKU'],
    how='left'   # use 'inner' if you want only matched records
)


In [22]:
merged_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,...,Invoice_Quantity,Invoice Rate,Invoice_Amount,invoice_amt_gst,SGST %,CGST %,IGST %,SGST Amt,CGST Amt,IGST Amt
0,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,0b2680cf-c341-4d7f-84f3-d6b78c86f7a5,48236900,810103154292,ECO SOUL | Disposable Paper Cups/Glasses | 180...,60.04,...,40.0,60.04,2401.6,2833.89,9.0,9.0,None,216.145,216.145,None
1,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,3ff23046-b471-475d-8927-8e30f726979a,46021919,810103152274,ECO SOUL | Palm Leaf Disposable Plates | 8 Inc...,83.85,...,80.0,83.85,6708.0,6708.00,0.0,0.0,None,0.000,0.000,None
2,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,50.0,69.11,3455.5,3628.28,2.5,2.5,None,86.390,86.390,None
3,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,958ba1c5-d0a9-465a-9e1b-925d3a9feeb3,48237010,810103158153,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,98.12,...,80.0,98.12,7849.6,8242.08,2.5,2.5,None,196.240,196.240,None
4,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,120.0,49.02,5882.4,6941.23,9.0,9.0,None,529.415,529.415,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1639,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,800e08df-4bb0-489e-9d3a-6e5f817cb919,48236900,810103156807,ECO SOUL | Disposable Paper Cups/Glasses | 180...,54.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1640,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1641,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1642,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,df871848-e955-4f51-8703-bb19ef5a5fe9,44219990,810103151604,Eco Soul 160 mm | 50 Count | Disposable Spoon ...,57.45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
final_df=merged_df.copy()

# Add platform and channel

In [24]:
final_df[['ClientID', 'Platform', 'Channel']] = [
    'TH-1755734939046', 'QuickCommerce', 'Zepto']

In [25]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,...,invoice_amt_gst,SGST %,CGST %,IGST %,SGST Amt,CGST Amt,IGST Amt,ClientID,Platform,Channel
0,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,0b2680cf-c341-4d7f-84f3-d6b78c86f7a5,48236900,810103154292,ECO SOUL | Disposable Paper Cups/Glasses | 180...,60.04,...,2833.89,9.0,9.0,None,216.145,216.145,None,TH-1755734939046,QuickCommerce,Zepto
1,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,3ff23046-b471-475d-8927-8e30f726979a,46021919,810103152274,ECO SOUL | Palm Leaf Disposable Plates | 8 Inc...,83.85,...,6708.00,0.0,0.0,None,0.000,0.000,None,TH-1755734939046,QuickCommerce,Zepto
2,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,3628.28,2.5,2.5,None,86.390,86.390,None,TH-1755734939046,QuickCommerce,Zepto
3,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,958ba1c5-d0a9-465a-9e1b-925d3a9feeb3,48237010,810103158153,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,98.12,...,8242.08,2.5,2.5,None,196.240,196.240,None,TH-1755734939046,QuickCommerce,Zepto
4,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,6941.23,9.0,9.0,None,529.415,529.415,None,TH-1755734939046,QuickCommerce,Zepto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1639,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,800e08df-4bb0-489e-9d3a-6e5f817cb919,48236900,810103156807,ECO SOUL | Disposable Paper Cups/Glasses | 180...,54.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TH-1755734939046,QuickCommerce,Zepto
1640,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TH-1755734939046,QuickCommerce,Zepto
1641,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TH-1755734939046,QuickCommerce,Zepto
1642,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,df871848-e955-4f51-8703-bb19ef5a5fe9,44219990,810103151604,Eco Soul 160 mm | 50 Count | Disposable Spoon ...,57.45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TH-1755734939046,QuickCommerce,Zepto


In [26]:
extra_columns = [
    'grn_item_code',
    'grn_product_description',
    'grn_mrp',
    'grn_tax_amount',
    'grn_quantity',
    'grn_total_amount',
    'grn_gmv_loss',
    'dn_date',
    'facility_name',
    'dn_id',
    'item_name',
    'vendor_invoice_id',
    'dn_quantity',
    'dn_value',
    'dn_reason',
    'dn_remark',
]

for col in extra_columns:
    final_df[col] = ""


# Add NLC

In [27]:
import pandas as pd

revenue_df = pd.read_excel(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Shambhavi's File\NLC All platform1.xlsx",sheet_name='Zepto',header=1)

# Rename columns
revenue_df = revenue_df.rename(columns={
    'SKUs': 'SKU',
    'MRP':'MRP as per pricing sheet',
    'GST':'GST as per pricing sheet',
})

# (Optional but recommended) keep only required columns
revenue_df = revenue_df[['SKU', 'NLC as per pricing sheet','MRP as per pricing sheet','GST as per pricing sheet']]

# Clean SKU
revenue_df['SKU'] = revenue_df['SKU'].astype(str).str.strip()

final_df = final_df.merge(
    revenue_df[['SKU', 'NLC as per pricing sheet','MRP as per pricing sheet','GST as per pricing sheet']],
    on='SKU',
    how='left'
)
missing_skus = final_df.loc[final_df['NLC as per pricing sheet'].isna(), 'SKU'].unique().tolist()
    

# Add revenue

In [28]:
import numpy as np

# Ensure numeric types (very important)
final_df['NLC as per pricing sheet'] = pd.to_numeric(final_df['NLC as per pricing sheet'], errors='coerce')
final_df['Invoice_Quantity'] = pd.to_numeric(final_df['Invoice_Quantity'], errors='coerce')

# Create revenue column
final_df['revenue'] = final_df['NLC as per pricing sheet'] * final_df['Invoice_Quantity']


# Add status from file

In [29]:
import pandas as pd

# Load zepto file
zepto_file = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Grijesh Gautam's files - Zepto(All)\ESH-Zepto Delivery Details.xlsx"

zepto_df = pd.read_excel(zepto_file, sheet_name="Appointment Dates",header =1)


# Select only required columns and rename them
zepto_selected = zepto_df[[
    'PO Number', 
    'Appt Date', 
    'PO Status', 
    'Req Appt Date'
]].copy()

# Rename columns as requested
zepto_selected = zepto_selected.rename(columns={
    'Appt Date': 'appt date', 
    'PO Status': 'PO_fulfilled_status_from_file',
    'Req Appt Date': 'requested_appt_date'
})
# Convert PO Number to string in both dataframes
final_df['PO Number'] = final_df['PO Number'].astype(str).str.strip()
zepto_selected['PO Number'] = zepto_selected['PO Number'].astype(str).str.strip()

# Merge with your existing final_df (left join on PO_Number)
final_df = final_df.merge(
    zepto_selected, 
    on='PO Number', 
    how='left'  # Keeps all rows from final_df
)

print("✅ Merge complete!")
print(f"Shape before: {len(final_df)} rows")
print(f"New columns added: appt date, PO_fulfilled_status_from_file, requested_appt_date")


✅ Merge complete!
Shape before: 1644 rows
New columns added: appt date, PO_fulfilled_status_from_file, requested_appt_date


# Add a status

In [30]:
import numpy as np
final_df["Invoice_Quantity"] = (
    final_df["Invoice_Quantity"]
    .astype(str)
    .str.split("\n")          # split multiline cell
    .str[0]                   # take first value (1120.0)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
)

final_df['PO_quantity'] = pd.to_numeric(
    final_df['PO_quantity'], errors='coerce'
).fillna(0)

final_df['Status'] = np.where(
    final_df['Invoice Number'].isna(),
    'Unfulfilled',
    np.where(final_df['Invoice_Quantity'] < final_df['PO_quantity'], 'Short Shipped', 'Fulfilled')
)


# Change datatypes

In [31]:
import pandas as pd

# ---------- TEXT columns ----------
text_cols = [
    'PO Number', 'PO Creation Date', 'PO Expiry Date',
    'PO Delivered By Address', 'PO Item Code', 'HSN Code',
    'PO Product UPC', 'PO Product Description',
    'PO Basic Cost Price', 'PO Landing Rate',
    'Invoice Number', 'Invoice Date', 'SKU',
    'Invoice Item & Description', 'Category', 'Material',
    'Sub Category', 'Status', 'ClientID', 'Platform',
    'Channel', 'PO MRP', 'Invoice Rate',
    'CGST %', 'SGST %', 'IGST %',
    'grn_product_description', 'grn_tax_amount',
    'grn_total_amount', 'grn_gmv_loss',
    'dn_date', 'facility_name', 'dn_id',
    'item_name', 'vendor_invoice_id',
    'dn_reason', 'dn_remark',
    'appt date', 'PO_fulfilled_status_from_file',
    'requested_appt_date'
]

for col in text_cols:
    final_df[col] = final_df[col].astype('string')


# ---------- BIGINT(20) columns ----------
bigint_cols = [
    'PO_quantity',
    'Box/Case',
    'grn_item_code',
    'grn_mrp',
    'grn_quantity',
    'MRP as per pricing sheet'
]

for col in bigint_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce').astype('Int64')


# ---------- DOUBLE columns ----------
double_cols = [
    'PO Tax Amt',
    'Invoice_Quantity',
    'PO_Amount',
    'Invoice_Amount',
    'invoice_amt_gst',
    'CGST Amt',
    'SGST Amt',
    'IGST Amt',
    'dn_quantity',
    'dn_value',
    'NLC as per pricing sheet',
    'GST as per pricing sheet',
    'revenue'
]

for col in double_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce').astype('float64')


# Add year, month

In [32]:
final_df['PO Creation Date'] = pd.to_datetime(
    final_df['PO Creation Date'],
    errors='coerce'
)
final_df['year_month'] = final_df['PO Creation Date'].dt.to_period('M').astype(str)
final_df['PO Creation Date'] = final_df['PO Creation Date'].dt.date

float_cols = final_df.select_dtypes(include='float')

final_df[float_cols.columns] = final_df[float_cols.columns].round(2)
import pandas as pd

final_df['year_month'] = pd.to_datetime(
    final_df['year_month'],
    errors='coerce'
)

final_df['year'] = final_df['year_month'].dt.year
final_df['month'] = final_df['year_month'].dt.strftime('%B')
# Convert columns to string type
final_df['year'] = final_df['year'].astype(str)
final_df['month'] = final_df['month'].astype(str)
final_df['year_month'] = final_df['year_month'].astype(str)


In [33]:
final_df['year'] = final_df['year'].astype(str)
final_df['month'] = final_df['month'].astype(str).str.zfill(2)

final_df['year_month'] = (
    final_df['year'] + '_' + final_df['month']
)


In [34]:
final_df['invoice_amt_gst'].sum()

np.float64(9697044.629999999)

In [35]:
final_df['PO_Amount'].sum()

np.float64(13592901.57)

In [36]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,...,MRP as per pricing sheet,GST as per pricing sheet,revenue,appt date,PO_fulfilled_status_from_file,requested_appt_date,Status,year_month,year,month
0,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,0b2680cf-c341-4d7f-84f3-d6b78c86f7a5,48236900,810103154292,ECO SOUL | Disposable Paper Cups/Glasses | 180...,60.04,...,149,0.18,2401.6,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
1,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,3ff23046-b471-475d-8927-8e30f726979a,46021919,810103152274,ECO SOUL | Palm Leaf Disposable Plates | 8 Inc...,83.85,...,169,0.00,6708.0,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
2,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,169,0.05,3455.5,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
3,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,958ba1c5-d0a9-465a-9e1b-925d3a9feeb3,48237010,810103158153,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,98.12,...,229,0.05,7849.6,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
4,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,89,0.18,5882.4,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1639,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,800e08df-4bb0-489e-9d3a-6e5f817cb919,48236900,810103156807,ECO SOUL | Disposable Paper Cups/Glasses | 180...,54.53,...,119,0.18,NaN,Active-Action Required,Active-Action Required,Active-Action Required,Unfulfilled,2026_June,2026,June
1640,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,169,0.05,NaN,Active-Action Required,Active-Action Required,Active-Action Required,Unfulfilled,2026_June,2026,June
1641,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,89,0.18,NaN,Active-Action Required,Active-Action Required,Active-Action Required,Unfulfilled,2026_June,2026,June
1642,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,df871848-e955-4f51-8703-bb19ef5a5fe9,44219990,810103151604,Eco Soul 160 mm | 50 Count | Disposable Spoon ...,57.45,...,149,0.05,NaN,Active-Action Required,Active-Action Required,Active-Action Required,Unfulfilled,2026_June,2026,June


In [37]:
final_df['PO_fulfilled_status_from_file'].unique()

<StringArray>
[             'Fulfilled',  'Appointment requested',  'Appointment Scheduled',
             'In Transit', 'Active-Action Required']
Length: 5, dtype: string

In [38]:
final_df = final_df[~final_df['PO_fulfilled_status_from_file']
                    .isin(['Did not deliver', 'Cancelled by Vendor'])]


In [39]:
final_df['invoice_amt_gst'].sum()

np.float64(9697044.629999999)

In [40]:
final_df["PO_Amount"].sum()

np.float64(13592901.57)

In [41]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,...,MRP as per pricing sheet,GST as per pricing sheet,revenue,appt date,PO_fulfilled_status_from_file,requested_appt_date,Status,year_month,year,month
0,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,0b2680cf-c341-4d7f-84f3-d6b78c86f7a5,48236900,810103154292,ECO SOUL | Disposable Paper Cups/Glasses | 180...,60.04,...,149,0.18,2401.6,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
1,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,3ff23046-b471-475d-8927-8e30f726979a,46021919,810103152274,ECO SOUL | Palm Leaf Disposable Plates | 8 Inc...,83.85,...,169,0.00,6708.0,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
2,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,169,0.05,3455.5,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
3,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,958ba1c5-d0a9-465a-9e1b-925d3a9feeb3,48237010,810103158153,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,98.12,...,229,0.05,7849.6,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
4,P3979255,2026-04-01,2026-04-29 23:59:00,ECOSOUL HOME PRIVATE LIMITED,CHN-DRY-MH2-THIRUVALLUR,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,89,0.18,5882.4,2026-04-27 00:00:00,Fulfilled,2026-04-16 00:00:00,Fulfilled,2026_April,2026,April
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1639,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,800e08df-4bb0-489e-9d3a-6e5f817cb919,48236900,810103156807,ECO SOUL | Disposable Paper Cups/Glasses | 180...,54.53,...,119,0.18,NaN,Active-Action Required,Active-Action Required,Active-Action Required,Unfulfilled,2026_June,2026,June
1640,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,810103158733,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,169,0.05,NaN,Active-Action Required,Active-Action Required,Active-Action Required,Unfulfilled,2026_June,2026,June
1641,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,810103151888,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,89,0.18,NaN,Active-Action Required,Active-Action Required,Active-Action Required,Unfulfilled,2026_June,2026,June
1642,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,df871848-e955-4f51-8703-bb19ef5a5fe9,44219990,810103151604,Eco Soul 160 mm | 50 Count | Disposable Spoon ...,57.45,...,149,0.05,NaN,Active-Action Required,Active-Action Required,Active-Action Required,Unfulfilled,2026_June,2026,June


# Save it locally

In [42]:
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta
import os

# =====================================================
# FILE PATH
# =====================================================
CSV_PATH = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\zepto_output.csv"

# =====================================================
# STEP 1: CALCULATE CUTOFF DATE (CURRENT MONTH - 2)
# =====================================================
today = date.today()

cutoff_date = (
    today.replace(day=1)
    - relativedelta(months=2)
)

print(f"🧹 CSV cleanup from: {cutoff_date}")

# =====================================================
# STEP 2: LOAD EXISTING CSV (IF PRESENT)
# =====================================================
if os.path.exists(CSV_PATH):
    existing_df = pd.read_csv(CSV_PATH)
    print(f"📂 Existing CSV rows: {len(existing_df)}")
else:
    print("⚠️ CSV not found. Creating new one.")
    existing_df = pd.DataFrame()

# =====================================================
# STEP 3: PARSE DATE COLUMN SAFELY
# =====================================================
if not existing_df.empty:
    existing_df["PO Creation Date"] = pd.to_datetime(
        existing_df["PO Creation Date"],
        errors="coerce"
    )

    before_rows = len(existing_df)

    # =================================================
    # STEP 4: REMOVE LAST 3-MONTH WINDOW DATA
    # =================================================
    existing_df = existing_df[
        existing_df["PO Creation Date"] < pd.Timestamp(cutoff_date)
    ]

    after_rows = len(existing_df)
    print(f"🗑️ Rows removed from CSV: {before_rows - after_rows}")

# =====================================================
# STEP 5: APPEND CURRENT FINAL_DF
# =====================================================
final_df = final_df.copy()

# Ensure date column is datetime
final_df["PO Creation Date"] = pd.to_datetime(
    final_df["PO Creation Date"],
    errors="coerce"
)

updated_df = pd.concat(
    [existing_df, final_df],
    ignore_index=True
)

print(f"➕ Rows added: {len(final_df)}")
print(f"📊 Final CSV rows: {len(updated_df)}")

# =====================================================
# STEP 6: SAVE BACK TO SAME LOCATION
# =====================================================
updated_df.to_csv(CSV_PATH, index=False)

print(f"💾 CSV updated successfully:\n{CSV_PATH}")


🧹 CSV cleanup from: 2026-04-01


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_21560\2884623355.py:27: DtypeWarning: Columns (34,35,39) have mixed types. Specify dtype option on import or set low_memory=False.
  existing_df = pd.read_csv(CSV_PATH)
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_21560\2884623355.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated_df = pd.concat(


📂 Existing CSV rows: 9233
🗑️ Rows removed from CSV: 1547
➕ Rows added: 1644
📊 Final CSV rows: 9330
💾 CSV updated successfully:
C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\zepto_output.csv


In [43]:
import pandas as pd
final_df = pd.read_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\zepto_output.csv")

C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_21560\2670660669.py:2: DtypeWarning: Columns (34,35,39) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\zepto_output.csv")


In [44]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,...,NLC as per pricing sheet,MRP as per pricing sheet,GST as per pricing sheet,revenue,appt date,PO_fulfilled_status_from_file,requested_appt_date,year_month,year,month
0,3100661896,2025-01-18,16-02-2025,"ECOSOUL HOME PRIVATE LIMITED Advant Navis, Tow...",KIRANAKART TECHNOLOGIES PVT LTD KIRANAKART TEC...,120235,39232990,8.101030e+11,Compostable Trash Bags Small - Set Of 15 1 pcs,43.52,...,32.50,119,0.18,0.0,03-02-2025 00:00,Fulfilled,03-02-2025 00:00,2025_January,2025,January
1,3100661896,2025-01-18,16-02-2025,"ECOSOUL HOME PRIVATE LIMITED Advant Navis, Tow...",KIRANAKART TECHNOLOGIES PVT LTD KIRANAKART TEC...,117178,48181000,8.101030e+11,Compostable Bagasse Facial Tissue - 2Ply Set O...,43.52,...,38.01,129,0.18,0.0,03-02-2025 00:00,Fulfilled,03-02-2025 00:00,2025_January,2025,January
2,3100661896,2025-01-18,16-02-2025,"ECOSOUL HOME PRIVATE LIMITED Advant Navis, Tow...",KIRANAKART TECHNOLOGIES PVT LTD KIRANAKART TEC...,120447,48236900,8.101030e+11,"Tree Free Paper Napkins, 2-Ply, 50 Count, Pack...",32.50,...,32.50,89,0.18,0.0,03-02-2025 00:00,Fulfilled,03-02-2025 00:00,2025_January,2025,January
3,3100661896,2025-01-18,16-02-2025,"ECOSOUL HOME PRIVATE LIMITED Advant Navis, Tow...",KIRANAKART TECHNOLOGIES PVT LTD KIRANAKART TEC...,116684,39232990,8.101030e+11,Compostable Trash Bags Medium - Set Of 15 1 pcs,49.02,...,38.01,129,0.18,0.0,03-02-2025 00:00,Fulfilled,03-02-2025 00:00,2025_January,2025,January
4,3100661896,2025-01-18,16-02-2025,"ECOSOUL HOME PRIVATE LIMITED Advant Navis, Tow...",KIRANAKART TECHNOLOGIES PVT LTD KIRANAKART TEC...,117327,48181000,8.101030e+11,Compostable Bagasse Kitchen Paper Towel - 2 Pl...,82.07,...,82.07,229,0.18,0.0,03-02-2025 00:00,Fulfilled,03-02-2025 00:00,2025_January,2025,January
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9325,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,800e08df-4bb0-489e-9d3a-6e5f817cb919,48236900,8.101032e+11,ECO SOUL | Disposable Paper Cups/Glasses | 180...,54.53,...,54.53,119,0.18,NaN,Active-Action Required,Active-Action Required,Active-Action Required,2026_June,2026,June
9326,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,94b0ae0e-75e3-4853-ab5b-b01ffbe1e2df,48237010,8.101032e+11,ECO SOUL | Bagasse Disposable Meal Tray/Plates...,69.11,...,69.11,169,0.05,NaN,Active-Action Required,Active-Action Required,Active-Action Required,2026_June,2026,June
9327,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,c5101eaa-acf1-4ae1-bf95-0b8b2fa9ef84,48236900,8.101032e+11,ECO SOUL | Disposable Paper Cups/Glasses | 120...,49.02,...,49.02,89,0.18,NaN,Active-Action Required,Active-Action Required,Active-Action Required,2026_June,2026,June
9328,P4614376,2026-06-11,2026-07-18 23:59:00,ECOSOUL HOME PRIVATE LIMITED,MUM-DRY-MH3,df871848-e955-4f51-8703-bb19ef5a5fe9,44219990,8.101032e+11,Eco Soul 160 mm | 50 Count | Disposable Spoon ...,57.45,...,57.45,149,0.05,NaN,Active-Action Required,Active-Action Required,Active-Action Required,2026_June,2026,June


In [45]:
import pandas as pd

# convert to datetime (handle mixed formats)
final_df['PO Creation Date'] = pd.to_datetime(final_df['PO Creation Date'], format='mixed')
final_df['PO Expiry Date'] = pd.to_datetime(final_df['PO Expiry Date'], format='mixed')

# extract date only
final_df['PO Creation Date'] = final_df['PO Creation Date'].dt.date
final_df['PO Expiry Date'] = final_df['PO Expiry Date'].dt.date

# extract year
final_df['year'] = pd.to_datetime(final_df['PO Creation Date']).dt.year

# extract month name
final_df['month'] = pd.to_datetime(final_df['PO Creation Date']).dt.month_name()

# create year_month column
final_df['year_month'] = final_df['year'].astype(str) + '_' + final_df['month']

In [46]:
final_df.columns

Index(['PO Number', 'PO Creation Date', 'PO Expiry Date',
       'PO Delivered By Address', 'PO Delivered To', 'PO Item Code',
       'HSN Code', 'PO Product UPC', 'PO Product Description',
       'PO Basic Cost Price', 'PO Tax Amt', 'PO Landing Rate',
       'Invoice Number', 'Invoice Date', 'invoice_bill_to', 'invoice_ship_to',
       'invoice_dispatch_from', 'SKU', 'Invoice Item & Description',
       'Category', 'Material', 'Sub Category', 'PO_quantity',
       'Invoice_Quantity', 'PO_Amount', 'Invoice_Amount', 'invoice_amt_gst',
       'Status', 'ClientID', 'Platform', 'Channel', 'PO MRP', 'Box/Case',
       'Invoice Rate', 'CGST %', 'SGST %', 'CGST Amt', 'SGST Amt', 'IGST Amt',
       'IGST %', 'grn_item_code', 'grn_product_description', 'grn_mrp',
       'grn_tax_amount', 'grn_quantity', 'grn_total_amount', 'grn_gmv_loss',
       'dn_date', 'facility_name', 'dn_id', 'item_name', 'vendor_invoice_id',
       'dn_quantity', 'dn_value', 'dn_reason', 'dn_remark',
       'NLC as per p

# Save it to Mysql

In [47]:
# import mysql.connector
# from sqlalchemy import create_engine
# from datetime import date
# from dateutil.relativedelta import relativedelta
# from urllib.parse import quote_plus
# import pandas as pd
# import re

# # =====================================================
# # COLUMN CLEANING FUNCTION (YOUR VERSION)
# # =====================================================
# def clean_mysql_column(col):
#     col = col.strip()
#     col = col.replace(" ", "_")
#     col = col.replace("%", "pct")
#     col = col.replace("-", "_")
#     col = col.replace("/", "_")
#     col = col.replace("\\", "_")
#     col = col.replace("(", "")
#     col = col.replace(")", "")
#     col = col.replace("[", "")
#     col = col.replace("]", "")
#     col = re.sub(r"__+", "_", col)
#     col = re.sub(r"[^A-Za-z0-9_]", "", col)
#     return col.lower()

# # =====================================================
# # MYSQL CONFIG
# # =====================================================
# MYSQL_CONFIG = {
#     "host": "192.168.50.148",
#     "port": 3306,
#     "user": "datahive_esh_test",
#     "password": "Datahive@321!",
#     "database": "datahive_esh_test",
# }

# DEPARTMENT = "quick_comm"
# BASE_TABLE_NAME = "zepto_orders"
# TABLE_NAME = f"{DEPARTMENT}_{BASE_TABLE_NAME}"

# # =====================================================
# # STEP 1: CALCULATE CUTOFF DATE (CURRENT MONTH - 2)
# # =====================================================
# today = date.today()

# cutoff_date = (
#     today.replace(day=1)
#     - relativedelta(months=2)
# )

# print(f"🧹 DB cleanup from: {cutoff_date}")

# # =====================================================
# # STEP 2: DELETE EXISTING DATA (DEC + JAN + FEB)
# # =====================================================
# conn = mysql.connector.connect(**MYSQL_CONFIG)
# cursor = conn.cursor()

# delete_query = f"""
# DELETE FROM {TABLE_NAME}
# WHERE po_creation_date >= %s
# """

# cursor.execute(delete_query, (cutoff_date,))
# deleted_rows = cursor.rowcount

# conn.commit()
# cursor.close()
# conn.close()

# print(f"✅ Deleted {deleted_rows} rows from DB")

# # =====================================================
# # STEP 3: CLEAN COLUMN NAMES USING YOUR FUNCTIONz``
# # =====================================================
# final_df.columns = [clean_mysql_column(c) for c in final_df.columns]

# # =====================================================
# # STEP 4: APPEND DATA TO MYSQL
# # =====================================================
# encoded_password = quote_plus(MYSQL_CONFIG["password"])

# engine = create_engine(
#     f"mysql+mysqlconnector://{MYSQL_CONFIG['user']}:{encoded_password}@"
#     f"{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}"
# )

# final_df.to_sql(
#     name=TABLE_NAME,
#     con=engine,
#     if_exists="append",
#     index=False,
#     chunksize=2000,
#     method="multi"
# )

# print(f"🚀 Inserted {len(final_df)} rows into {TABLE_NAME}")


In [48]:
final_df = pd.read_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\zepto_output.csv")

C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_21560\2948103373.py:1: DtypeWarning: Columns (34,35,39) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\zepto_output.csv")


In [49]:
import pandas as pd
import urllib.parse
import re
from sqlalchemy import create_engine, text, inspect

# ======================================================
# MYSQL CONFIG
# ======================================================
MYSQL_CONFIG = {
    "host": "192.168.50.148",
    "port": 3306,
    "user": "datahive_esh_test",
    "password": "Datahive@321!",
    "database": "datahive_esh_test",
}

DEPARTMENT = "quick_comm"
BASE_TABLE_NAME = "zepto_orders"
TABLE_NAME = f"{DEPARTMENT}_{BASE_TABLE_NAME}"

# ======================================================
# CLEAN MYSQL COLUMN NAMES
# ======================================================
def clean_mysql_column(col):
    col = col.strip()
    col = col.replace(" ", "_")
    col = col.replace("%", "pct")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace("\\", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("[", "")
    col = col.replace("]", "")
    col = re.sub(r"__+", "_", col)
    col = re.sub(r"[^A-Za-z0-9_]", "", col)
    return col.lower()

# ======================================================
# CREATE MYSQL ENGINE
# ======================================================
password_encoded = urllib.parse.quote_plus(MYSQL_CONFIG["password"])

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_CONFIG['user']}:{password_encoded}"
    f"@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}",
    pool_pre_ping=True
)

# ======================================================
# TEST MYSQL CONNECTION (OPTIONAL BUT RECOMMENDED)
# ======================================================
def test_mysql_connection():
    print("\n🔍 Testing MySQL connection...")
    try:
        with engine.connect() as conn:
            db = conn.execute(text("SELECT DATABASE();")).fetchone()
            print("✅ Connected to database:", db[0])
    except Exception as e:
        print("❌ Connection failed:", e)
        return False
    return True

# ======================================================
# SAVE final_df TO MYSQL
# ======================================================
def save_final_df_to_mysql(final_df: pd.DataFrame):
    print("\n🟦 Uploading final_df to MySQL...")

    df = final_df.copy()
    print("📊 Rows to upload:", len(df))

    # Clean column names
    df.columns = [clean_mysql_column(c) for c in df.columns]

    inspector = inspect(engine)
    table_exists = inspector.has_table(TABLE_NAME)

    try:
        with engine.begin() as conn:

            if table_exists:
                print("🗑 Truncating existing table...")
                conn.execute(text(f"TRUNCATE TABLE `{TABLE_NAME}`"))
                mode = "append"
            else:
                print("📌 Creating new table...")
                mode = "fail"

            print("⬆ Inserting rows...")
            df.to_sql(
                TABLE_NAME,
                con=conn,
                if_exists=mode,
                index=False,
                chunksize=1000,
                method="multi"
            )

        print("✅ Upload completed successfully!")

    except Exception as e:
        print("❌ Upload failed:", e)

# ======================================================
# RUN
# ======================================================
# final_df MUST already exist in memory

if test_mysql_connection():
    save_final_df_to_mysql(final_df)



🔍 Testing MySQL connection...
✅ Connected to database: datahive_esh_test

🟦 Uploading final_df to MySQL...
📊 Rows to upload: 9330
🗑 Truncating existing table...
⬆ Inserting rows...
✅ Upload completed successfully!
